# YouTube Content Predictor - Colab Deployment

**Instructions:**
1. Upload the `youtube_project.zip` file (which we generated locally) into the files section on the left sidebar of this Colab notebook.
2. Run the cells below sequentially to extract the project, install dependencies, and spin up both the Flask backend and the React frontend.
3. Click the generated Localtunnel links at the bottom to view your app!

In [ ]:
# Extract the uploaded project zip
!unzip -o -q youtube_project.zip -d youtube_project
print("Extraction complete!")

In [ ]:
# Install dependencies
!pip install -r youtube_project/backend/requirements.txt
!npm install -g localtunnel
import os
os.chdir('/content/youtube_project/frontend')
!npm install

In [ ]:
import os
import time
import subprocess
import re

# 1. Start Flask Backend
os.chdir('/content/youtube_project/backend')
backend_process = subprocess.Popen(["python", "app.py"])
time.sleep(3)
print("Backend Flask server started.")

# 2. Expose Backend via localtunnel
os.system('nohup lt --port 5000 > backend_tunnel.log 2>&1 &')
time.sleep(3)

# 3. Extract the Backend URL
with open('backend_tunnel.log', 'r') as f:
    log = f.read()
backend_url = re.search(r'(https://[\w\.-]+loca\.lt)', log).group(1)
print("Backend Public URL is:", backend_url)

# 4. Patch the React App to use the public backend URL instead of localhost
app_jsx_path = '/content/youtube_project/frontend/src/App.jsx'
with open(app_jsx_path, 'r') as f:
    app_code = f.read()
app_code = app_code.replace('http://127.0.0.1:5000', backend_url)
with open(app_jsx_path, 'w') as f:
    f.write(app_code)
print("Patched React frontend to point to public backend API!")

In [ ]:
import urllib.request

# Start React Frontend (Vite)
os.chdir('/content/youtube_project/frontend')
frontend_process = subprocess.Popen(["npm", "run", "dev", "--", "--host", "0.0.0.0"])
time.sleep(5)

# Expose Frontend via localtunnel
os.system('nohup lt --port 5173 > frontend_tunnel.log 2>&1 &')
time.sleep(3)

ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("\n=================================================================")
print("IMPORTANT: Copy this password. You will need to enter it when opening the links.")
print("PASSWORD:", ip)
print("=================================================================\n")

print("Frontend Tunnel Log (CLICK THIS URL TO VIEW THE APP!):")
!cat frontend_tunnel.log